> 📝 **Planning note (author use -- remove before publishing)**
>
> **Intended contents:** load two real moorings (AT200 + AT800), combine onto a common time base, compare visually and via scatter plot, Pearson correlation.

**To do:** ~~fix the SyntaxError from prose sitting in a code cell~~ (done); ~~remove the `xr.corr??` debugging leftover~~ (already gone); ~~fix "PLot" typo~~ (done); fill in bullet placeholders; tie in interpolation-awareness from notebook 6 (what `combine_datasets` is doing under the hood when aligning two different sampling intervals). Also: resolve the two-correlations issue flagged below.

<img src="../../data/images/hiaoos_learning_moored.png" width="300" align="right">


# Comparing and correlating data from different instruments

We often want to compare or correlate records from different instruments -- for example, two moorings at different locations, or sensors at different depths. Before we can do that, we usually need to combine the datasets onto a common time base first.

So: combining is a means to an end here, not the point itself -- the goal is the comparison/correlation that follows.

In [ ]:
from kval.data import moored
import xarray as xr
import matplotlib.pyplot as plt
%matplotlib widget

## Combining the two records

- 

> ✏️ **Review note** — this now points at `AT200_21_22_SBE37_15252_113m_edited.nc`, **which does not exist yet** -- notebook 4 only processes the RBR/AT800 record. See the note at the end of notebook 4. Until that file is produced, this cell will fail.

Why it matters here and not just tidiness: the `_from_raw` AT200 file still contains on-deck measurements at both ends (`PRES` near $-$0.4 dbar, `TEMP` near 10 °C at the end of the record). Any statistic computed over it is partly a statistic about air.

Note also that the two records don't cover the same period -- AT800 ends in April 2022, AT200 runs to October 2022. `combine_datasets` handles this by filling NaN, and `xr.corr` drops non-overlapping pairs, so the correlation below is computed over the overlap only. Worth stating explicitly in the text.

In [ ]:
ds1 = moored.load_nc('../../data/moored_CTD_test_data/intermediate_data/AT200_21_22_SBE37_15252_113m_edited.nc')
ds2 = moored.load_nc('../../data/moored_CTD_test_data/intermediate_data/AT800_21_22_CONCERTO_60595_99m_edited.nc')

In [ ]:
ds_combined = moored.combine_datasets(ds1, ds2, interval = '1D', instr_names = ['AT200', 'AT800'])

In [ ]:
ds_combined

## Comparing and correlating the two records

- 
- 
- 

## Plot together

In [ ]:
moored.plot(ds_combined)

## Scatter plot


In [ ]:
TEMP_AT200 = ds_combined.TEMP.sel(INSTR = 'AT200')
TEMP_AT800 = ds_combined.TEMP.sel(INSTR = 'AT800')

In [ ]:

fig, ax = plt.subplots()
ax.scatter(TEMP_AT200, TEMP_AT800, s = 5, color = 'r')
ax.set_aspect('equal')
ax.grid()
ax.axline((-0.7, -0.7), (4.5, 4.5), color = 'k', ls = '--')
ax.set_xlabel('TEMP, AT200 [°C]')
ax.set_ylabel('TEMP, AT800 [°C]')

## Correlation

 Pearson correlation coefficient of the two time series 

In [ ]:
xr.corr(TEMP_AT200, TEMP_AT800)

> ✏️ **Review note** — these next two cells were sitting further up, under "Comparing and correlating", computing the same quantity a second way. I've moved them here so the two routes sit side by side, because they don't give the same answer: the combined-dataset route gave 0.5953 and this one gave 0.5918.

That isn't noise. `combine_datasets` anchors every dataset to a *shared* time origin so the daily bins line up exactly across instruments; calling `time_average` separately on each dataset lets each one anchor to its own first timestamp, so the two sets of daily means are averaging over slightly offset windows.

Pick one. Either drop these cells, or keep them and make the discrepancy the point -- it's a good, concrete illustration of why "put them on a common time base" is a real step and not bookkeeping.

In [ ]:
ds1_daily = moored.time_average(ds1, '1D')
ds2_daily = moored.time_average(ds2, '1D')

In [ ]:
xr.corr(ds1_daily.TEMP, ds2_daily.TEMP)